In [2]:
import os
os.chdir("/home/jovyan/work/MST")  # adjust if needed
os.getcwd()



'/home/jovyan/work/MST'

In [3]:
import torch
from pathlib import Path

from mst.models.dino import DinoV2ClassifierSlice
from mst_xai.xai_methods.gradcam_slice_level import GradCAM_Slice
from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

/home/jovyan/work/MST/mst/models/extern/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/jovyan/work/MST/mst/models/extern/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/jovyan/work/MST/mst/models/extern/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [4]:
ckpt_dir = Path("runs/ODELIA/DinoV2ClassifierSlice_Final")
model = DinoV2ClassifierSlice.load_best_checkpoint(ckpt_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
/opt/conda/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEn

DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [5]:
print(model)


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [6]:
model_structure = str(model)
print(model_structure)

DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [7]:
ds = ODELIA_Dataset3D(split="test")

sample = ds[0]   # single sample
uid = sample["uid"]
gt = int(sample["target"])

print("UID:", uid, "GT:", gt)


UID: ODELIA_BRAID1_0246_1_left GT: 0


In [8]:
batch = {
    "source": sample["source"].unsqueeze(0).to(device),
    "target": torch.tensor([gt], device=device)
}


In [9]:
batch["source"].shape
# expected: [1, 3, D, H, W]


torch.Size([1, 1, 32, 224, 224])

In [14]:
logits = model(batch["source"])
pred = logits.argmax(dim=1).item()

print("Predicted class:", pred)


Predicted class: 0


In [15]:
gradcam = GradCAM_Slice(model)


In [16]:
sal = gradcam.generate(batch, target_class=pred)


In [18]:
# Inside GradCAM_Slice, but we test externally by re-running backward
model

logits = model(batch["source"])
score = logits[:, pred].sum()
score.backward()

acts = gradcam.activations          # [B, N, C]
grads = gradcam.gradients           # [B, N, C]

print("CLS grad sum:", grads[:, 0, :].abs().sum().item())
print("Slice grad sum:", grads[:, 1:, :].abs().sum().item())


CLS grad sum: 9.631288528442383
Slice grad sum: 0.0


# Summary
The classifier depends ONLY on the CLS token.
Slice tokens receive ZERO gradient.

In [19]:
print("CAM shape:", sal.shape)
print("min / max / std:", sal.min(), sal.max(), sal.std())


CAM shape: torch.Size([32])
min / max / std: tensor(0., grad_fn=<MinBackward1>) tensor(0., grad_fn=<MaxBackward1>) tensor(0., grad_fn=<StdBackward0>)
